# Week 49, Day 1: Review: linear structures

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w049d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
nums = [2, 7, 11, 15, 1, 8]
target = 9

seen = {}
for i, x in enumerate(nums):
    if target - x in seen:
        print(seen[target - x], i)
    seen[x] = i

In [ ]:
def longest_unique(s):
    """Sliding window: the longest run with no repeated character."""
    last, left, best = {}, 0, 0
    for right, ch in enumerate(s):
        if ch in last and last[ch] >= left:
            left = last[ch] + 1               # jump past the repeat
        last[ch] = right
        best = max(best, right - left + 1)
    return best

def count_sum(nums, k):
    """Prefix sums with a dict: subarrays summing to k."""
    counts, total, found = {0: 1}, 0, 0
    for x in nums:
        total += x
        found += counts.get(total - k, 0)
        counts[total] = counts.get(total, 0) + 1
    return found

print(longest_unique("abcabcbb"), longest_unique("pwwkew"), count_sum([1, 1, 1], 2), count_sum([1, 2, 3, -3, 3], 3))

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def build(values):
    head = None
    for v in reversed(values):
        head = Node(v, head)
    return head

def to_list(head):
    out = []
    while head:
        out.append(head.value)
        head = head.next
    return out

def remove_kth_from_end(head, k):
    dummy = Node(None, head)
    fast = slow = dummy
    for _ in range(k):
        fast = fast.next
    while fast.next:
        fast, slow = fast.next, slow.next
    slow.next = slow.next.next
    return dummy.next

def reverse(head):
    prev = None
    while head:
        head.next, prev, head = prev, head, head.next
    return prev

print(to_list(remove_kth_from_end(build([1, 2, 3, 4, 5]), 2)), to_list(remove_kth_from_end(build([1]), 1)), to_list(reverse(build([1, 2, 3]))))

In [ ]:
def next_greater(nums):
    answer, stack = [-1] * len(nums), []       # stack of indices still waiting
    for i, x in enumerate(nums):
        while stack and nums[stack[-1]] < x:
            answer[stack.pop()] = x
        stack.append(i)
    return answer

def balanced(s):
    pairs, stack = {")": "(", "]": "[", "}": "{"}, []
    for ch in s:
        if ch in "([{":
            stack.append(ch)
        elif ch in pairs:
            if not stack or stack.pop() != pairs[ch]:
                return False
    return not stack

print(next_greater([2, 1, 2, 4, 3]), balanced("{[()()]}"), balanced("([)]"))

In [ ]:
from collections import Counter, defaultdict

def group_anagrams(words):
    groups = defaultdict(list)
    for w in words:
        groups["".join(sorted(w))].append(w)
    return list(groups.values())

def first_unique(s):
    counts = Counter(s)
    return next((i for i, ch in enumerate(s) if counts[ch] == 1), -1)

print(group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"]), first_unique("loveleetcode"))

> **Key idea.** Two pointers work on sorted arrays or for in-place filtering; a sliding window grows right and shrinks left so each index moves twice; prefix sums answer range sums in O(1), and with a dict count subarrays with a given sum. Linked lists use dummy heads, fast and slow pointers, and careful saving of links. Stacks match and undo; queues serve in order; monotonic stacks and deques give next-greater answers and window maxima in O(n). Dicts and sets answer "seen it?", "how many?", "grouped by what?" and "which prefix?" in O(1) average. Name the pattern before writing code.

### Exercise: Pairs with a difference

Write `count_pairs_with_difference(nums, k)` for a **sorted** list with no duplicates and k > 0: the number of pairs `(i, j)` with `nums[j] - nums[i] == k`, using two pointers moving in the **same** direction (O(n), no sets or dicts). Then `closest_pair_sum(nums, target)`: the pair sum closest to `target` (the smaller sum on a tie), with two pointers from both ends. `ValueError` if fewer than two numbers.

In [ ]:
def count_pairs_with_difference(nums, k):
    ___

def closest_pair_sum(nums, target):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_pairs():\n    \"pairs\"\n    assert count_pairs_with_difference([1, 3, 5, 7, 8], 2) == 3 and count_pairs_with_difference([1, 2], 5) == 0\n    assert closest_pair_sum([1, 4, 6, 9], 12) == 13 and closest_pair_sum([1, 4, 6, 9], 11) == 10 and closest_pair_sum([1, 3], 100) == 4\n    try:\n        closest_pair_sum([1], 2)\n        assert False\n    except ValueError:\n        pass\n    rng = random.Random(1)\n    for _ in range(200):\n        nums = sorted(rng.sample(range(60), rng.randint(2, 15)))\n        k, t = rng.randint(1, 10), rng.randint(0, 120)\n        assert count_pairs_with_difference(nums, k) == sum(1 for a in nums for b in nums if b - a == k)\n        sums = [a + b for i, a in enumerate(nums) for b in nums[i + 1:]]\n        assert closest_pair_sum(nums, t) == min(sums, key=lambda s: (abs(s - t), s))\n    assert \"set(\" not in source and \"dict(\" not in source and \"{\" not in source")

### Exercise: At most k kinds

Write `longest_with_k_kinds(items, k)`: the length of the longest contiguous run containing at most k **distinct** values, with a sliding window and a `Counter` (remove a value from the counter when its count drops to 0). Then `shortest_with_all(items)`: the length of the shortest contiguous run containing every distinct value in `items` (0 for an empty list).

In [ ]:
from collections import Counter

def longest_with_k_kinds(items, k):
    ___

def shortest_with_all(items):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_window():\n    \"windows\"\n    assert longest_with_k_kinds(\"eceba\", 2) == 3 and longest_with_k_kinds(\"aa\", 1) == 2 and longest_with_k_kinds(\"abc\", 0) == 0\n    assert shortest_with_all(\"aabcbcdbca\") == 4 and shortest_with_all([]) == 0 and shortest_with_all([5]) == 1\n    rng = random.Random(2)\n    for _ in range(200):\n        s = [rng.randint(0, 4) for _ in range(rng.randint(0, 20))]\n        k = rng.randint(0, 4)\n        subs = [s[i:j] for i in range(len(s)) for j in range(i, len(s) + 1)]\n        assert longest_with_k_kinds(s, k) == max((len(x) for x in subs if len(set(x)) <= k), default=0)\n        kinds = len(set(s))\n        assert shortest_with_all(s) == min((len(x) for x in subs if len(set(x)) == kinds), default=0)")

### Exercise: Rotating and reordering

For singly linked lists of `Node`s, write `rotate_right(head, k)`: move the last k nodes (k may exceed the length) to the front, relinking, in O(n); and `reorder(head)`: turn `a1, a2, ..., an` into `a1, an, a2, an-1, ...` in place (find the middle with fast and slow pointers, reverse the second half, then interleave). Both return the new head.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def rotate_right(head, k):
    ___

def reorder(head):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def build(values):\n    head = None\n    for v in reversed(values):\n        head = Node(v, head)\n    return head\ndef to_list(head):\n    out = []\n    while head:\n        out.append(head.value)\n        head = head.next\n        assert len(out) < 1000, \"a cycle\"\n    return out\ndef test_linked():\n    \"relinking\"\n    assert to_list(rotate_right(build([1, 2, 3, 4, 5]), 2)) == [4, 5, 1, 2, 3]\n    assert to_list(rotate_right(build([0, 1, 2]), 4)) == [2, 0, 1] and rotate_right(None, 3) is None and to_list(rotate_right(build([1]), 9)) == [1]\n    assert to_list(reorder(build([1, 2, 3, 4]))) == [1, 4, 2, 3] and to_list(reorder(build([1, 2, 3, 4, 5]))) == [1, 5, 2, 4, 3]\n    assert reorder(None) is None and to_list(reorder(build([7]))) == [7]\n    for n in range(0, 12):\n        vals = list(range(n))\n        want = []\n        i, j = 0, n - 1\n        while i <= j:\n            want.append(vals[i])\n            if i != j:\n                want.append(vals[j])\n            i, j = i + 1, j - 1\n        assert to_list(reorder(build(vals))) == want\n        for k in range(0, 3 * n + 1):\n            assert to_list(rotate_right(build(vals), k)) == (vals[-(k % n):] + vals[:-(k % n)] if n and k % n else vals)")

### Exercise: Spans and temperatures

Write `days_until_warmer(temps)`: for each day, how many days until a strictly warmer one (0 if never), and `price_spans(prices)`: for each day, how many consecutive days ending that day (including it) had a price less than or equal to that day's. Both with monotonic stacks in O(n).

In [ ]:
def days_until_warmer(temps):
    ___

def price_spans(prices):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef test_stacks():\n    \"monotonic stacks\"\n    assert days_until_warmer([73, 74, 75, 71, 69, 72, 76, 73]) == [1, 1, 4, 2, 1, 1, 0, 0]\n    assert price_spans([100, 80, 60, 70, 60, 75, 85]) == [1, 1, 1, 2, 1, 4, 6]\n    rng = random.Random(3)\n    for _ in range(200):\n        xs = [rng.randint(0, 9) for _ in range(rng.randint(0, 15))]\n        assert days_until_warmer(xs) == [next((j - i for j in range(i + 1, len(xs)) if xs[j] > xs[i]), 0) for i in range(len(xs))]\n        spans = []\n        for i in range(len(xs)):\n            k = i\n            while k >= 0 and xs[k] <= xs[i]:\n                k -= 1\n            spans.append(i - k)\n        assert price_spans(xs) == spans\n    big = list(range(100_000, 0, -1))\n    t0 = time.perf_counter()\n    days_until_warmer(big)\n    price_spans(list(range(100_000)))\n    assert time.perf_counter() - t0 < 2")

### Exercise: Prefix sums with a dict

Write `count_divisible(nums, k)`: how many non-empty contiguous subarrays have a sum divisible by k (k > 0), in one pass with a dict counting prefix sums **modulo k**. Then `longest_balanced(bits)`: the length of the longest contiguous run with equally many 0s and 1s, by treating 0 as −1 and remembering the **first** position of each prefix sum.

In [ ]:
def count_divisible(nums, k):
    ___

def longest_balanced(bits):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_prefix():\n    \"prefix sums\"\n    assert count_divisible([4, 5, 0, -2, -3, 1], 5) == 7 and count_divisible([5], 9) == 0\n    assert longest_balanced([0, 1, 0]) == 2 and longest_balanced([0, 1, 1, 0, 1, 1, 1, 0]) == 4 and longest_balanced([1, 1]) == 0\n    rng = random.Random(4)\n    for _ in range(200):\n        xs = [rng.randint(-5, 5) for _ in range(rng.randint(0, 15))]\n        k = rng.randint(1, 6)\n        assert count_divisible(xs, k) == sum(1 for i in range(len(xs)) for j in range(i + 1, len(xs) + 1) if sum(xs[i:j]) % k == 0)\n        bits = [rng.randint(0, 1) for _ in range(rng.randint(0, 15))]\n        assert longest_balanced(bits) == max([j - i for i in range(len(bits)) for j in range(i + 1, len(bits) + 1) if bits[i:j].count(0) * 2 == j - i] + [0])")

### Bug hunt: A window that never shrinks enough

This sliding window shrinks only once per step (`if` instead of `while`), so after a long run of repeats the window still contains a duplicate, and the answer is too big: this prints `5`. Shrink until the window is valid again, so it prints `4`.

In [ ]:
from collections import Counter

def longest_without_repeats(s):
    counts, left, best = Counter(), 0, 0
    for right, ch in enumerate(s):
        counts[ch] += 1
        if counts[ch] > 1:
            counts[s[left]] -= 1
            left += 1
        best = max(best, right - left + 1)
    return best

print(longest_without_repeats("abcdbx"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"shrink fully\"\n    expect_output(\"4\")\n    assert longest_without_repeats(\"abcabcbb\") == 3")

### Bug hunt: Lost the rest of the list

This reversal changes `node.next` before saving it, so the walk stops after the first node and the rest of the list is lost: this prints `[1]`. Save the next node first so it prints `[3, 2, 1]`.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def reverse(head):
    prev, node = None, head
    while node:
        node.next = prev
        prev = node
        node = node.next
    return prev

node, out = reverse(Node(1, Node(2, Node(3)))), []
while node:
    out.append(node.value)
    node = node.next
print(out)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"save the link\"\n    expect_output(\"[3, 2, 1]\")")

### Bug hunt: Forgetting the empty prefix

This counts subarrays summing to k with prefix sums, but starts the dict empty instead of with `{0: 1}`, so subarrays that start at the very beginning are never counted: this prints `1`. Seed the dict so it prints `2`.

In [ ]:
def count_sum(nums, k):
    counts, total, found = {}, 0, 0
    for x in nums:
        total += x
        found += counts.get(total - k, 0)
        counts[total] = counts.get(total, 0) + 1
    return found

print(count_sum([1, 1, 1], 2))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the empty prefix\"\n    expect_output(\"2\")")

### Puzzle: Next greater element

Arrange the lines into the monotonic-stack "next greater" function. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
from collections import deque, Counter

p = ___   # the prefix sums of [3, 1, 4, 1], starting with 0
q = ___   # d = deque([1, 2, 3]); d.appendleft(0); d.pop(); list(d)
r = ___   # Counter("banana").most_common(1)
s = ___   # the next greater element list of [4, 5, 2, 25]
t = ___   # the middle node's value found by fast and slow pointers (slow one step, fast two, both from the head) in 1 -> 2 -> 3 -> 4 -> 5
u = ___   # the cost of inserting at the front of a Python list of n items

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [0, 3, 4, 8, 9] and q == [0, 1, 2] and r == [(\"a\", 3)], \"p, q and r\"\n    assert s == [5, 25, 25, -1] and t == 3 and u.replace(\" \", \"\") == \"O(n)\", \"s, t and u\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a rate limiter

A web service allows each user at most `limit` requests in any `window` seconds. Write `RateLimiter(limit, window)` with `allow(user, t)`: called with non-decreasing times `t` (whole seconds), it returns `True` and records the request if the user has made fewer than `limit` **allowed** requests in the half-open period `(t - window, t]`, and `False` otherwise (refused requests aren't recorded). Keep a `deque` of allowed times per user in a dict, dropping times that have left the window from the front. Also `active_users(t)`: how many users have at least one allowed request in `(t - window, t]`, and `forget_idle(t)`: delete the records of users with none, returning how many were deleted. Each `allow` must be O(1) amortised.

In [ ]:
from collections import deque

class RateLimiter:
    def __init__(self, limit, window):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef test_limiter():\n    \"limits\"\n    r = RateLimiter(3, 10)\n    assert [r.allow(\"ana\", t) for t in [0, 1, 2, 3, 9, 10, 11, 12, 13]] == [True, True, True, False, False, True, True, True, False]\n    assert r.allow(\"bo\", 13) and r.active_users(13) == 2\n    assert r.forget_idle(20) == 0 and r.active_users(30) == 0 and r.forget_idle(30) == 2\ndef test_random():\n    \"against a slow version\"\n    rng = random.Random(5)\n    limit, window = 4, 7\n    r, log, t = RateLimiter(limit, window), {}, 0\n    for _ in range(3000):\n        t += rng.randint(0, 2)\n        u = rng.choice(\"abcde\")\n        recent = [x for x in log.get(u, []) if t - window < x <= t]\n        want = len(recent) < limit\n        assert r.allow(u, t) == want\n        if want:\n            log.setdefault(u, []).append(t)\n        if rng.random() < 0.05:\n            assert r.active_users(t) == sum(1 for v in log.values() if any(t - window < x <= t for x in v))\ndef test_speed():\n    \"O(1) amortised\"\n    r = RateLimiter(1000, 50)\n    t0 = time.perf_counter()\n    for t in range(200_000):\n        r.allow(\"one\", t // 3)\n    assert time.perf_counter() - t0 < 3")

## How to review

The same rules as before, because they're the ones that work:

1. Try every prediction **before** running it, and write your guess down.
2. When a guess is wrong, find exactly why before moving on.
3. Keep a list of everything that surprised you. That list is your revision plan for this week's timed sets.

One new habit for this week: before writing any code for a problem, **name the pattern** ("sliding window", "monotonic stack", "prefix sums with a dict") and say what it would cost. If you can't name one, write the brute-force version first, then ask which repeated work it does.

## Arrays and their patterns

From Week 41, three patterns turn O(n²) array problems into O(n):

- **Two pointers** (Week 41 Day 3): on a **sorted** array, one pointer at each end, moving whichever the comparison says; or a slow writer and a fast reader for in-place filtering.
- **Sliding window** (Week 41 Day 4): a window `[left, right]` that grows on the right and shrinks on the left while a condition fails; each index enters and leaves once.
- **Prefix sums** (Week 41 Day 5): `prefix[i]` is the sum of the first i items, so any range sum is one subtraction; with a dict counting earlier prefix values, "how many subarrays sum to k" becomes one pass.

Behind them all: a dynamic array (Python's `list`) appends in O(1) amortised, but inserting or deleting at the front shifts everything, O(n).

## Linked lists

Week 42's rules for linked lists:

- A **dummy head** removes the special case for changing the first node.
- **Fast and slow pointers** find the middle (fast moves two, slow one), the kth from the end (fast starts k ahead), and cycles (Floyd).
- **Reversal** needs three names: `prev`, `node` and the saved `node.next`.
- Draw the pointers before and after; most bugs are a link changed before it was saved.

The tuple assignment in `reverse` works because Python evaluates the whole right-hand side first, then assigns left to right; `head.next` is assigned before `head` changes.

## Stacks and queues

Week 43's toolkit:

- A **stack** (a list, `append` and `pop`) for anything that must be matched or undone in reverse: brackets, undo, depth-first search, evaluating expressions.
- A **queue** (`collections.deque`, `append` and `popleft`) for first come, first served: breadth-first search, buffers, scheduling.
- A **monotonic stack** answers "the next greater (or smaller) element" for every position in O(n): keep a stack of indices whose answers aren't known yet, in decreasing order of value; a new value pops (and answers) every smaller one.
- A **monotonic deque** gives the maximum of every sliding window in O(n).

## Hash tables

Week 44: dicts and sets give O(1) average lookups because a hash function picks the slot, collisions are handled (by chaining or open addressing), and the table grows to keep the load factor low. In problems, a dict is usually doing one of four jobs:

- **Have I seen this?** A set (the warm-up's two-sum used a dict for "where did I see it?").
- **How many of each?** A `Counter`.
- **Group by a key**: `dict.setdefault(key, []).append(item)` or `defaultdict(list)`; anagrams group by `"".join(sorted(word))`.
- **Remember a prefix**: prefix sums, prefix XORs or prefix states, counted or with their first position.

Keys must be hashable and must not change while in the table.

## Choosing a structure

| you need | use | cost |
|---|---|---|
| index by position, append | `list` | O(1), O(1) amortised |
| add and remove at both ends | `deque` | O(1) |
| membership, counting, grouping | `set`, `dict`, `Counter` | O(1) average |
| insert or delete in the middle, given the node | linked list | O(1) |
| last in, first out | `list` as a stack | O(1) |
| first in, first out | `deque` as a queue | O(1) |
| "next greater", window maxima | monotonic stack or deque | O(n) for all |
| range sums | prefix sums | O(n) set-up, O(1) per query |

## Practice

## Quiz

## Daily challenge

## Before you go

Look back at your list of surprises from today. For each, write one sentence saying what you'll look for next time. Tomorrow: the same for trees, heaps and strings.